# Phase 9: Time-Series Analysis

Weekly seasonal decomposition, month-of-year seasonality, and year-over-year comparisons of daily revenue.

## Setup

In [1]:
import pandas as pd, numpy as np, json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

df = pd.read_csv('../data/polokwane_sales_clean.csv', parse_dates=['date'])
sales = df[df['value_zar'] > 0].copy()
out = '../outputs'

daily = sales.groupby('date')['value_zar'].sum().asfreq('D').fillna(0)

# Weekly seasonal decomposition (period=7)
decomp = seasonal_decompose(daily, model='additive', period=7, extrapolate_trend='freq')
fig = decomp.plot()
fig.set_size_inches(10, 8)
plt.tight_layout()
plt.savefig(f'{out}/seasonal_decompose_weekly.png', dpi=150)
plt.close()

# Monthly aggregation + month-of-year seasonality (avg DAILY revenue across years)
monthly = sales.groupby(sales['date'].dt.to_period('M'))['value_zar'].sum()
daily_df = daily.reset_index()
daily_df.columns = ['date', 'daily_revenue']
daily_df['month'] = daily_df['date'].dt.month
month_of_year = daily_df.groupby('month')['daily_revenue'].mean()

fig, ax = plt.subplots(figsize=(8,4.5))
month_of_year.plot(kind='bar', ax=ax, color='#2563eb')
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=0)
ax.set_title('Average Daily Revenue by Month of Year')
ax.set_ylabel('Avg daily revenue (ZAR)')
plt.tight_layout()
plt.savefig(f'{out}/seasonality_month_of_year.png', dpi=150)
plt.close()

# Week-over-week growth rate
weekly = sales.groupby(sales['date'].dt.to_period('W'))['value_zar'].sum()
wow_growth = weekly.pct_change().dropna()

# Year-over-year comparison (matching months across years)
yoy = sales.groupby([sales['date'].dt.year, sales['date'].dt.month])['value_zar'].sum().unstack(0)
yoy.to_csv(f'{out}/yoy_monthly_revenue.csv')

fig, ax = plt.subplots(figsize=(9,5))
yoy.plot(ax=ax, marker='o')
ax.set_title('Year-over-Year Monthly Revenue Comparison')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (ZAR)')
ax.set_xticks(range(1,13))
plt.tight_layout()
plt.savefig(f'{out}/yoy_comparison.png', dpi=150)
plt.close()

strongest_month = month_of_year.idxmax()
weakest_month = month_of_year.idxmin()
avg_wow_vol = wow_growth.std()

summary = {
    'strongest_month_of_year': int(strongest_month),
    'weakest_month_of_year': int(weakest_month),
    'week_over_week_growth_volatility_std': float(avg_wow_vol),
    'seasonal_component_range': [float(decomp.seasonal.min()), float(decomp.seasonal.max())],
}
with open(f'{out}/timeseries_summary.json','w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("\nMonth-of-year avg daily revenue:")
print(month_of_year.round(0))

/tmp/ipykernel_675/2565114677.py:14: FutureWarning: `extrapolate_trend='freq'` is deprecated and will be removed in 0.16, use `extrapolate_trend='period'` instead.
  decomp = seasonal_decompose(daily, model='additive', period=7, extrapolate_trend='freq')


{
  "strongest_month_of_year": 12,
  "weakest_month_of_year": 6,
  "week_over_week_growth_volatility_std": 0.34431982309743775,
  "seasonal_component_range": [
    -73314.59895106222,
    35023.94350525557
  ]
}

Month-of-year avg daily revenue:
month
1      89773.0
2      85297.0
3      81824.0
4      84151.0
5      81943.0
6      77727.0
7      89977.0
8      85259.0
9      80585.0
10     80757.0
11     80104.0
12    104238.0
Name: daily_revenue, dtype: float64
